In [169]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Credit Risk Assessment

In [170]:
import pandas as pd
import numpy as np

import src.utils as utils

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

## Business Understanding

blabla

## Objective Metrics

blabla

## Data Preparation

In [171]:
df = pd.read_csv('input/loan_data_2007_2014.csv', low_memory=False)

utils.skim_data(df)

Total duplicate rows: 0
DF shape: (466285, 75)


,feature,dtype,null_%,negative_%,zero_%,n_unique,unique_%,sample_values
0,Unnamed: 0,int64,0.000,0.0,0.0,466285,100.00,"[0, 1, 2, 3, 4]"
1,id,int64,0.000,0.0,0.0,466285,100.00,"[1077501, 1077430, 1077175, 1076863, 1075358]"
2,member_id,int64,0.000,0.0,0.0,466285,100.00,"[1296599, 1314167, 1313524, 1277178, 1311748]"
3,loan_amnt,int64,0.000,0.0,0.0,1352,0.29,"[5000, 2500, 2400, 10000, 3000]"
4,funded_amnt,int64,0.000,0.0,0.0,1354,0.29,"[5000, 2500, 2400, 10000, 3000]"
5,funded_amnt_inv,float64,0.000,0.0,0.05,9854,2.11,"[4975.0, 2500.0, 2400.0, 10000.0, 3000.0]"
6,term,object,0.000,-,-,2,0.00,"[ 36 months, 60 months]"
7,int_rate,float64,0.000,0.0,0.0,506,0.11,"[10.65, 15.27, 15.96, 13.49, 12.69]"
8,installment,float64,0.000,0.0,0.0,55622,11.93,"[162.87, 59.83, 84.33, 339.31, 67.79]"
9,grade,object,0.000,-,-,7,0.00,"[B, C, A, E, F]"


Berdasarkan hasil `skim_data`, terdapat 466.285 baris data dan 75 kolom fitur. Kolom targetnya adalah `loan_status`. Dari jumlah baris sebanyak itu, tidak terlihat ada data duplikat. Meskipun demikian, 75 fitur ini masih dapat dikurangi untuk mempermudah eksplorasi dan membuat model jadi lebih efisien dan akurat. Ada lima langkah yang bisa saya lakukan untuk mengurangi jumlah fitur ini, yaitu:

- Filter berdasarkan identitas dan informasi administratif: membuang fitur seperti nama dan ID nasabah, karena fitur seperti itu tidak memiliki nilai prediktif.
- Filter berdasarkan kualitas data: membuang (a) fitur yang memiliki *missing values* yang terlalu banyak dan (b) fitur yang hanya memiliki satu nilai unik karena fitur ini tidak memberikan informasi pembeda bagi model.
- Filter berdasarkan fitur yang berpotensi menimbulkan *data leakage*.
- Filter berdasarkan fitur yang memiliki variasi rendah.
- Filter berdasarkan fitur yang memiliki jumlah nilai 0 yang tinggi.

Selain mengurangi jumlah fitur, saya juga perlu mengubah fitur-fitur kategorikal yang bisa diubah menjadi fitur numerik.

### Identifying IDs and Adminstrative Informations

Fitur-fitur yang tergolong dalam kategori ini adalah `id`, `member_id`, `url`, `desc`, `policy_code`, `Unnamed: 0`, `emp_title`, `title`, dan `zip_code`.

In [172]:
df = df.drop(columns=['id', 'member_id', 'url', 'desc', 'policy_code', 'Unnamed: 0', 'emp_title', 'title', 'zip_code'])
skim_result = utils.skim_data(df)

skim_result

Total duplicate rows: 0
DF shape: (466285, 66)


,feature,dtype,null_%,negative_%,zero_%,n_unique,unique_%,sample_values
0,loan_amnt,int64,0.000,0.0,0.0,1352,0.29,"[5000, 2500, 2400, 10000, 3000]"
1,funded_amnt,int64,0.000,0.0,0.0,1354,0.29,"[5000, 2500, 2400, 10000, 3000]"
2,funded_amnt_inv,float64,0.000,0.0,0.05,9854,2.11,"[4975.0, 2500.0, 2400.0, 10000.0, 3000.0]"
3,term,object,0.000,-,-,2,0.00,"[ 36 months, 60 months]"
4,int_rate,float64,0.000,0.0,0.0,506,0.11,"[10.65, 15.27, 15.96, 13.49, 12.69]"
5,installment,float64,0.000,0.0,0.0,55622,11.93,"[162.87, 59.83, 84.33, 339.31, 67.79]"
6,grade,object,0.000,-,-,7,0.00,"[B, C, A, E, F]"
7,sub_grade,object,0.000,-,-,35,0.01,"[B2, C4, C5, C1, B5]"
8,emp_length,object,4.505,-,-,11,0.00,"[10+ years, < 1 year, 1 year, 3 years, 8 years]"
9,home_ownership,object,0.000,-,-,6,0.00,"[RENT, OWN, MORTGAGE, OTHER, NONE]"


### Low-Quality Features

Ada dua hal yang harus dilakukan dalam tahap ini:

- Menghapus fitur yang memiliki *missing values* > 50%, dan
- Menghapus fitur yang hanya memiliki satu nilai.

In [173]:
null_features = skim_result[skim_result['null_%'] > 50]['feature'].tolist()
print(f'Null features: {null_features}')
df = df.drop(columns=null_features)
skim_result = utils.skim_data(df) # update the skim result

skim_result

Null features: ['mths_since_last_delinq', 'mths_since_last_record', 'mths_since_last_major_derog', 'annual_inc_joint', 'dti_joint', 'verification_status_joint', 'open_acc_6m', 'open_il_6m', 'open_il_12m', 'open_il_24m', 'mths_since_rcnt_il', 'total_bal_il', 'il_util', 'open_rv_12m', 'open_rv_24m', 'max_bal_bc', 'all_util', 'inq_fi', 'total_cu_tl', 'inq_last_12m']
Total duplicate rows: 0
DF shape: (466285, 46)


,feature,dtype,null_%,negative_%,zero_%,n_unique,unique_%,sample_values
0,loan_amnt,int64,0.000,0.0,0.0,1352,0.29,"[5000, 2500, 2400, 10000, 3000]"
1,funded_amnt,int64,0.000,0.0,0.0,1354,0.29,"[5000, 2500, 2400, 10000, 3000]"
2,funded_amnt_inv,float64,0.000,0.0,0.05,9854,2.11,"[4975.0, 2500.0, 2400.0, 10000.0, 3000.0]"
3,term,object,0.000,-,-,2,0.00,"[ 36 months, 60 months]"
4,int_rate,float64,0.000,0.0,0.0,506,0.11,"[10.65, 15.27, 15.96, 13.49, 12.69]"
5,installment,float64,0.000,0.0,0.0,55622,11.93,"[162.87, 59.83, 84.33, 339.31, 67.79]"
6,grade,object,0.000,-,-,7,0.00,"[B, C, A, E, F]"
7,sub_grade,object,0.000,-,-,35,0.01,"[B2, C4, C5, C1, B5]"
8,emp_length,object,4.505,-,-,11,0.00,"[10+ years, < 1 year, 1 year, 3 years, 8 years]"
9,home_ownership,object,0.000,-,-,6,0.00,"[RENT, OWN, MORTGAGE, OTHER, NONE]"


Terdapat 20 fitur yang memiliki proporsi *missing value* di atas 50%, dan semuanya telah dihapus dari data. Sekarang mari lihat fitur yang hanya memiliki satu nilai.

In [174]:
one_value_features = skim_result[skim_result['n_unique'] == 1]['feature'].tolist()
df = df.drop(columns=one_value_features)
skim_result = utils.skim_data(df)

skim_result

Total duplicate rows: 0
DF shape: (466285, 45)


,feature,dtype,null_%,negative_%,zero_%,n_unique,unique_%,sample_values
0,loan_amnt,int64,0.000,0.0,0.0,1352,0.29,"[5000, 2500, 2400, 10000, 3000]"
1,funded_amnt,int64,0.000,0.0,0.0,1354,0.29,"[5000, 2500, 2400, 10000, 3000]"
2,funded_amnt_inv,float64,0.000,0.0,0.05,9854,2.11,"[4975.0, 2500.0, 2400.0, 10000.0, 3000.0]"
3,term,object,0.000,-,-,2,0.00,"[ 36 months, 60 months]"
4,int_rate,float64,0.000,0.0,0.0,506,0.11,"[10.65, 15.27, 15.96, 13.49, 12.69]"
5,installment,float64,0.000,0.0,0.0,55622,11.93,"[162.87, 59.83, 84.33, 339.31, 67.79]"
6,grade,object,0.000,-,-,7,0.00,"[B, C, A, E, F]"
7,sub_grade,object,0.000,-,-,35,0.01,"[B2, C4, C5, C1, B5]"
8,emp_length,object,4.505,-,-,11,0.00,"[10+ years, < 1 year, 1 year, 3 years, 8 years]"
9,home_ownership,object,0.000,-,-,6,0.00,"[RENT, OWN, MORTGAGE, OTHER, NONE]"


Hanya terdapat satu fitur yang memiliki satu nilai, yaitu `application_type`. Fitur tersebut telah dihapus.

### Transforming non-numerical data

In [175]:
categorical_cols = skim_result[skim_result['dtype'] == 'object']['feature'].tolist()
df[categorical_cols].sample(10)

,term,grade,sub_grade,emp_length,home_ownership,verification_status,issue_d,loan_status,pymnt_plan,purpose,addr_state,earliest_cr_line,initial_list_status,last_pymnt_d,next_pymnt_d,last_credit_pull_d
293050,36 months,A,A1,10+ years,MORTGAGE,Source Verified,Oct-14,Current,n,debt_consolidation,AL,Jan-96,f,Jan-16,Feb-16,Jan-16
213012,60 months,E,E5,5 years,MORTGAGE,Verified,Jun-12,Fully Paid,n,debt_consolidation,PA,Oct-00,f,Apr-13,NaN,Jul-15
371698,36 months,A,A4,10+ years,MORTGAGE,Not Verified,Jun-14,Fully Paid,n,debt_consolidation,PA,Feb-00,w,Jul-14,NaN,Sep-15
405069,60 months,D,D2,10+ years,MORTGAGE,Verified,Apr-14,Fully Paid,n,debt_consolidation,NY,Mar-89,w,Jan-16,NaN,Jan-16
343731,60 months,E,E2,10+ years,MORTGAGE,Not Verified,Jul-14,Fully Paid,n,debt_consolidation,WV,Mar-93,w,Dec-15,NaN,Dec-15
119988,36 months,A,A4,10+ years,MORTGAGE,Verified,Jul-13,Current,n,debt_consolidation,MA,Jul-96,f,Jan-16,Feb-16,Jan-16
100419,36 months,A,A1,3 years,MORTGAGE,Source Verified,Aug-13,Fully Paid,n,debt_consolidation,GA,Sep-81,f,Oct-15,NaN,Jan-16
117121,36 months,B,B1,6 years,MORTGAGE,Not Verified,Jul-13,Fully Paid,n,debt_consolidation,CA,Jul-98,f,Jan-14,NaN,Jan-14
139242,36 months,C,C5,10+ years,MORTGAGE,Verified,May-13,Current,n,credit_card,MA,Oct-88,f,Jan-16,Jan-16,Jan-16
344188,36 months,B,B1,8 years,MORTGAGE,Not Verified,Jul-14,Current,n,credit_card,AR,Jun-06,w,Jan-16,Feb-16,Jan-16


Terdapat satu fitur kategorikal yang dapat diubah menjadi biner secara numerik, yaitu `pymnt_plan`.

#### `pymnt_plan`

In [176]:
df['pymnt_plan_flag'] = df['pymnt_plan'].map({'y': 1, 'n': 0})

display(df[['pymnt_plan', 'pymnt_plan_flag']].sample(10))

,pymnt_plan,pymnt_plan_flag
192778,n,0
223598,n,0
260074,n,0
393283,n,0
431340,n,0
111032,n,0
361658,n,0
348702,n,0
247997,n,0
392818,n,0


Jangan lupa untuk mengubah fitur-fitur yang bisa jadi tipe data `date`, seperti `issue_d` dan `earliest_cr_line`.

In [177]:
df['earliest_cr_line_date'] = pd.to_datetime(df['earliest_cr_line'], format='%b-%y')
df['issue_d_date'] = pd.to_datetime(df['issue_d'], format='%b-%y')

display(df[['issue_d', 'issue_d_date', 'earliest_cr_line', 'earliest_cr_line_date']].sample(10))

,issue_d,issue_d_date,earliest_cr_line,earliest_cr_line_date
441971,Feb-14,2014-02-01,Jul-91,1991-07-01
190528,Oct-12,2012-10-01,Nov-87,1987-11-01
416502,Apr-14,2014-04-01,Oct-03,2003-10-01
203753,Aug-12,2012-08-01,Sep-97,1997-09-01
400601,Apr-14,2014-04-01,Feb-77,1977-02-01
271120,Oct-14,2014-10-01,Sep-02,2002-09-01
392213,May-14,2014-05-01,Sep-87,1987-09-01
2564,Nov-11,2011-11-01,May-88,1988-05-01
459706,Jan-14,2014-01-01,Sep-03,2003-09-01
351799,Jul-14,2014-07-01,Dec-93,1993-12-01


Semua fitur yang telah diubah dapat dihapus untuk menghindari redundansi.

In [178]:
unused_features = ['pymnt_plan', 'issue_d', 'earliest_cr_line']
df = df.drop(columns=unused_features)

# rename the converted features to its original names
df = df.rename(columns={
    'pymnt_plan_flag': 'pymnt_plan',
})

display(
    df[['pymnt_plan']].sample(10)
)

,pymnt_plan
215493,0
246599,0
298579,0
167423,0
166310,0
57687,0
349445,0
188231,0
173640,0
240996,0


In [179]:
skim_result = utils.skim_data(df)

skim_result

Total duplicate rows: 0
DF shape: (466285, 45)


,feature,dtype,null_%,negative_%,zero_%,n_unique,unique_%,sample_values
0,loan_amnt,int64,0.000,0.0,0.0,1352,0.29,"[5000, 2500, 2400, 10000, 3000]"
1,funded_amnt,int64,0.000,0.0,0.0,1354,0.29,"[5000, 2500, 2400, 10000, 3000]"
2,funded_amnt_inv,float64,0.000,0.0,0.05,9854,2.11,"[4975.0, 2500.0, 2400.0, 10000.0, 3000.0]"
3,term,object,0.000,-,-,2,0.00,"[ 36 months, 60 months]"
4,int_rate,float64,0.000,0.0,0.0,506,0.11,"[10.65, 15.27, 15.96, 13.49, 12.69]"
5,installment,float64,0.000,0.0,0.0,55622,11.93,"[162.87, 59.83, 84.33, 339.31, 67.79]"
6,grade,object,0.000,-,-,7,0.00,"[B, C, A, E, F]"
7,sub_grade,object,0.000,-,-,35,0.01,"[B2, C4, C5, C1, B5]"
8,emp_length,object,4.505,-,-,11,0.00,"[10+ years, < 1 year, 1 year, 3 years, 8 years]"
9,home_ownership,object,0.000,-,-,6,0.00,"[RENT, OWN, MORTGAGE, OTHER, NONE]"


### High Probability of Data Leakage

Dalam konteks credit risk assessment, fitur-fitur ini dianggap sebagai `data leakage` karena fitur tersebut mengandung informasi masa depan yang tidak tersedia saat pengambilan keputusan kredit ketika aplikasi itu diajukan. Fitur-fitur tersebut adalah:

- `out_prncp`
- `out_prncp_inv`
- `total_pymnt`
- `total_pymnt_inv`
- `total_rec_prncp`
- `total_rec_int`
- `total_rec_late_fee`
- `recoveries`
- `collection_recovery_fee`
- `last_pymnt_d`
- `last_pymnt_amnt`
- `next_pymnt_d`
- `last_credit_pull_d`

In [180]:
potential_leak_features = ['out_prncp', 'out_prncp_inv', 'total_pymnt', 'total_pymnt_inv',
                           'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee',
                           'recoveries', 'collection_recovery_fee', 'last_pymnt_d',
                           'last_pymnt_amnt', 'next_pymnt_d', 'last_credit_pull_d']
df = df.drop(columns=potential_leak_features)
skim_result = utils.skim_data(df)

skim_result

Total duplicate rows: 0
DF shape: (466285, 32)


,feature,dtype,null_%,negative_%,zero_%,n_unique,unique_%,sample_values
0,loan_amnt,int64,0.000,0.0,0.0,1352,0.29,"[5000, 2500, 2400, 10000, 3000]"
1,funded_amnt,int64,0.000,0.0,0.0,1354,0.29,"[5000, 2500, 2400, 10000, 3000]"
2,funded_amnt_inv,float64,0.000,0.0,0.05,9854,2.11,"[4975.0, 2500.0, 2400.0, 10000.0, 3000.0]"
3,term,object,0.000,-,-,2,0.00,"[ 36 months, 60 months]"
4,int_rate,float64,0.000,0.0,0.0,506,0.11,"[10.65, 15.27, 15.96, 13.49, 12.69]"
5,installment,float64,0.000,0.0,0.0,55622,11.93,"[162.87, 59.83, 84.33, 339.31, 67.79]"
6,grade,object,0.000,-,-,7,0.00,"[B, C, A, E, F]"
7,sub_grade,object,0.000,-,-,35,0.01,"[B2, C4, C5, C1, B5]"
8,emp_length,object,4.505,-,-,11,0.00,"[10+ years, < 1 year, 1 year, 3 years, 8 years]"
9,home_ownership,object,0.000,-,-,6,0.00,"[RENT, OWN, MORTGAGE, OTHER, NONE]"


### Low-variance Features

Dari semua fitur yang telah disaring dan dibuat baru, mari cek `skim_result` terkini. Ada tiga fitur yang memiliki nilai 0 lebih dari 99% data, yaitu `collections_12_mths_ex_med`, `acc_now_delinq`, dan `pymnt_plan`. Ini menunjukkan kalau ketiga fitur tersebut bersifat *low-variance* sehingga tidak bermanfaat dalam pembuatan model.

In [181]:
low_var_features = ['collections_12_mths_ex_med', 'acc_now_delinq', 'pymnt_plan']
df = df.drop(columns=low_var_features)
skim_result = utils.skim_data(df)

skim_result

Total duplicate rows: 0
DF shape: (466285, 29)


,feature,dtype,null_%,negative_%,zero_%,n_unique,unique_%,sample_values
0,loan_amnt,int64,0.000,0.0,0.0,1352,0.29,"[5000, 2500, 2400, 10000, 3000]"
1,funded_amnt,int64,0.000,0.0,0.0,1354,0.29,"[5000, 2500, 2400, 10000, 3000]"
2,funded_amnt_inv,float64,0.000,0.0,0.05,9854,2.11,"[4975.0, 2500.0, 2400.0, 10000.0, 3000.0]"
3,term,object,0.000,-,-,2,0.00,"[ 36 months, 60 months]"
4,int_rate,float64,0.000,0.0,0.0,506,0.11,"[10.65, 15.27, 15.96, 13.49, 12.69]"
5,installment,float64,0.000,0.0,0.0,55622,11.93,"[162.87, 59.83, 84.33, 339.31, 67.79]"
6,grade,object,0.000,-,-,7,0.00,"[B, C, A, E, F]"
7,sub_grade,object,0.000,-,-,35,0.01,"[B2, C4, C5, C1, B5]"
8,emp_length,object,4.505,-,-,11,0.00,"[10+ years, < 1 year, 1 year, 3 years, 8 years]"
9,home_ownership,object,0.000,-,-,6,0.00,"[RENT, OWN, MORTGAGE, OTHER, NONE]"


### Redundant Features

Selanjutnya, terdapat masalah redundansi antara `grade` dan `sub_grade` dalam arti informasinya; `sub_grade` sudah mencakup `grade` dengan informasi level yang lebih detil. Saya akan memilih `sub_grade` karena level detil informasinya lebih tinggi dibanding `grade`, dengan harapan dapat meningkatkan performa dari model.

In [182]:
print(f'Grade values: {df['grade'].unique()}')
print(f'\nSubgrade values: {df['sub_grade'].unique()}')

Grade values: ['B' 'C' 'A' 'E' 'F' 'D' 'G']

Subgrade values: ['B2' 'C4' 'C5' 'C1' 'B5' 'A4' 'E1' 'F2' 'C3' 'B1' 'D1' 'A1' 'B3' 'B4'
 'C2' 'D2' 'A3' 'A5' 'D5' 'A2' 'E4' 'D3' 'D4' 'F3' 'E3' 'F4' 'F1' 'E5'
 'G4' 'E2' 'G3' 'G2' 'G1' 'F5' 'G5']


In [183]:
df = df.drop(columns=['grade'])
skim_result = utils.skim_data(df)

skim_result

Total duplicate rows: 0
DF shape: (466285, 28)


,feature,dtype,null_%,negative_%,zero_%,n_unique,unique_%,sample_values
0,loan_amnt,int64,0.000,0.0,0.0,1352,0.29,"[5000, 2500, 2400, 10000, 3000]"
1,funded_amnt,int64,0.000,0.0,0.0,1354,0.29,"[5000, 2500, 2400, 10000, 3000]"
2,funded_amnt_inv,float64,0.000,0.0,0.05,9854,2.11,"[4975.0, 2500.0, 2400.0, 10000.0, 3000.0]"
3,term,object,0.000,-,-,2,0.00,"[ 36 months, 60 months]"
4,int_rate,float64,0.000,0.0,0.0,506,0.11,"[10.65, 15.27, 15.96, 13.49, 12.69]"
5,installment,float64,0.000,0.0,0.0,55622,11.93,"[162.87, 59.83, 84.33, 339.31, 67.79]"
6,sub_grade,object,0.000,-,-,35,0.01,"[B2, C4, C5, C1, B5]"
7,emp_length,object,4.505,-,-,11,0.00,"[10+ years, < 1 year, 1 year, 3 years, 8 years]"
8,home_ownership,object,0.000,-,-,6,0.00,"[RENT, OWN, MORTGAGE, OTHER, NONE]"
9,annual_inc,float64,0.001,0.0,0.0,31901,6.84,"[24000.0, 30000.0, 12252.0, 49200.0, 80000.0]"


### Defining "Good Loan" vs. "Bad Loan"

Tujuan utama model yang akan dibuat adalah untuk menjawal pertanyaan bisnis sederhana: "apakah nasabah ini akan Gagal Bayar di masa depan", yang membuat permasalahan ini menjadi masalah klasifikasi biner.

Dalam `loan_status` sekarang, ada sembilan nilai unik yang harus dipilah agar dapat menjadi 1 (Gagal Bayar) atau 0 (Sukses Bayar):

In [184]:
df['loan_status'].unique()

array(['Fully Paid', 'Charged Off', 'Current', 'Default',
       'Late (31-120 days)', 'In Grace Period', 'Late (16-30 days)',
       'Does not meet the credit policy. Status:Fully Paid',
       'Does not meet the credit policy. Status:Charged Off'],
      dtype=object)

Untuk memilih kelompok Gagal Bayar, saya akan memilih semua variasi `Charged Off`, `Default`, dan variasi `Late (x days)`, karena saya ingin model ini secara proaktif mengidentifikasi calon nasabah yang berpotensi menyebabkan kerugian finansial bagi perusahaan.

Untuk memilih kelompok Sukses Bayar, saya akan memilih semua variasi `Fully Paid`, karena ini adalah mayoritas nasabah yang baik dan menguntungkan. Model harus bisa membedakan mereka dari kelompok berisiko.

Baris data yang tidak tergolong dalam dua kelompok di atas akan dibuang.

In [185]:
print(f'DataFrame sizes before selection: {df.shape}\n')
good_loan_statuses = [
    'Fully Paid', 
    'Does not meet the credit policy. Status:Fully Paid'
]
bad_loan_statuses = [
    'Charged Off', 
    'Default', 
    'Late (31-120 days)', 
    'Late (16-30 days)',
    'Does not meet the credit policy. Status:Charged Off'
]
df = df[df['loan_status'].isin(good_loan_statuses + bad_loan_statuses)].copy()
df['loan_status_binary'] = df['loan_status'].apply(
    lambda x: 1 if x in bad_loan_statuses else 0
)
# remove loan_status and replace with loan_status_binary
df = df.drop(columns=['loan_status'])
df = df.rename(columns={'loan_status_binary': 'loan_status'})
skim_result = utils.skim_data(df)

skim_result

DataFrame sizes before selection: (466285, 28)

Total duplicate rows: 0
DF shape: (238913, 28)


,feature,dtype,null_%,negative_%,zero_%,n_unique,unique_%,sample_values
0,loan_amnt,int64,0.000,0.0,0.0,1310,0.55,"[5000, 2500, 2400, 10000, 3000]"
1,funded_amnt,int64,0.000,0.0,0.0,1313,0.55,"[5000, 2500, 2400, 10000, 3000]"
2,funded_amnt_inv,float64,0.000,0.0,0.098,9560,4.00,"[4975.0, 2500.0, 2400.0, 10000.0, 5000.0]"
3,term,object,0.000,-,-,2,0.00,"[ 36 months, 60 months]"
4,int_rate,float64,0.000,0.0,0.0,505,0.21,"[10.65, 15.27, 15.96, 13.49, 7.9]"
5,installment,float64,0.000,0.0,0.0,43848,18.35,"[162.87, 59.83, 84.33, 339.31, 156.46]"
6,sub_grade,object,0.000,-,-,35,0.01,"[B2, C4, C5, C1, A4]"
7,emp_length,object,3.861,-,-,11,0.00,"[10+ years, < 1 year, 3 years, 9 years, 4 years]"
8,home_ownership,object,0.000,-,-,6,0.00,"[RENT, OWN, MORTGAGE, OTHER, NONE]"
9,annual_inc,float64,0.002,0.0,0.0,18715,7.83,"[24000.0, 30000.0, 12252.0, 49200.0, 36000.0]"


Terlihat ada pengurangan sebanyak 226.721 baris data dari `loan_status` berlabel `Current` dan `In Grace Period`. Sekilas ini terlihat seperti pemborosan data, tapi saya berpegang teguh bahwa kualitas data lebih penting daripada kuantitas data. Data tersebut adalah noise, dan mempertahankan noise akan berakibat fatal yang dapat merusak kemampuan model untuk belajar.

## Splitting Data

In [186]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['loan_status'])
y = df[['loan_status']]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=29, stratify=y
)

In [187]:
print(f'X_train shape: {X_train.shape}')
print(f'X_test shape: {X_test.shape}')

X_train shape: (191130, 27)
X_test shape: (47783, 27)


In [188]:
df_train = pd.concat([X_train, y_train], axis=1)
df_train.to_parquet('input/df_train.parquet', index=False)
df_test = pd.concat([X_test, y_test], axis=1)
df_test.to_parquet('input/df_test.parquet', index=False)